# Results Comparison and Analysis
**Wilson-Cowan Network Dynamics: Criticality & Higher-Order Interactions**

Comprehensive visualization and statistical analysis across all model variants:
- **3-node models:** 2nd-order additive/diffusive, 3rd-order additive/diffusive
- **10-node Ring Small-World:** 2nd-order additive, 3rd-order additive
- **10-node Random (Erdős–Rényi):** 2nd-order additive, 3rd-order additive

**Metrics:** oscillation_map, osc_fraction, TC, DTC, TC_ksg, DTC_ksg, Oinfo_ksg, cumulant, powercorr, skewness, kurtosis

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap, BoundaryNorm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
from pathlib import Path
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──
RESULTS_DIR = Path('../results')

# ── 3-Node model tags (files with KSG metrics) ──
MODELS_3N = {
    '2nd-Order Additive':  'Pairwise_noise',            # has KSG ✓
    '2nd-Order Diffusive': 'Pairwise_diffusive_noise',   # has KSG ✓
    '3rd-Order Additive':  'Coupling_noise',             # NO KSG (only Gaussian)
    '3rd-Order Diffusive': 'Coupling_diffusive_noise',   # has KSG ✓
}

# ── 10-Node model tags ──
MODELS_10N_RING = {
    'Ring 2nd Additive':  '10n_ring_sw_pairwise_noise',
    'Ring 2nd Diffusive': '10n_ring_sw_pairwise_diffusive_noise',
    'Ring 3rd Additive':  '10n_ring_sw_coupling_noise',
    'Ring 3rd Diffusive': '10n_ring_sw_coupling_diffusive_noise',
}

MODELS_10N_RANDOM = {
    'Random 2nd Additive':  '10n_random_pairwise_noise',
    'Random 2nd Diffusive': '10n_random_pairwise_diffusive_noise',
    'Random 3rd Additive':  '10n_random_coupling_noise',
    'Random 3rd Diffusive': '10n_random_coupling_diffusive_noise',
}

print('Configuration loaded.')

/var/folders/f5/3603lrxn363_ddpmt3c65sfm0000gn/T/ipykernel_34357/3588281369.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Configuration loaded.


In [2]:
def load_npz(tag):
    path = RESULTS_DIR / f'metrics_{tag}.npz'
    if not path.exists():
        print(f'  ⚠ NOT FOUND: {path.name}')
        return None
    d = dict(np.load(path))
    print(f'  ✓ {path.name}  keys={sorted(d.keys())}')
    return d

# ── Load 3-node ──
print('\n── 3-Node Models ──')
data_3n = {}
for name, tag in MODELS_3N.items():
    r = load_npz(tag)
    if r is not None:
        data_3n[name] = r

# ── Load 10-node Ring ──
print('\n── 10-Node Ring Small-World ──')
data_10n_ring = {}
for name, tag in MODELS_10N_RING.items():
    r = load_npz(tag)
    if r is not None:
        data_10n_ring[name] = r

# ── Load 10-node Random ──
print('\n── 10-Node Random ──')
data_10n_random = {}
for name, tag in MODELS_10N_RANDOM.items():
    r = load_npz(tag)
    if r is not None:
        data_10n_random[name] = r

print(f'\nLoaded: {len(data_3n)} 3N, {len(data_10n_ring)} 10N-Ring, {len(data_10n_random)} 10N-Random')


── 3-Node Models ──
  ✓ metrics_Pairwise_noise.npz  keys=['DTC', 'DTC_ksg', 'K3', 'Oinfo_ksg', 'P', 'TC', 'TC_ksg', 'cumulant', 'kurt', 'oscillation_map', 'powercorr', 'skew']
  ✓ metrics_Pairwise_diffusive_noise.npz  keys=['DTC', 'DTC_ksg', 'K3', 'Oinfo_ksg', 'P', 'TC', 'TC_ksg', 'cumulant', 'kurt', 'oscillation_map', 'powercorr', 'skew']
  ✓ metrics_Coupling_noise.npz  keys=['DTC', 'K3', 'P', 'TC', 'cumulant', 'kurt', 'oscillation_map', 'powercorr', 'skew']
  ✓ metrics_Coupling_diffusive_noise.npz  keys=['DTC', 'DTC_ksg', 'K3', 'Oinfo_ksg', 'P', 'TC', 'TC_ksg', 'cumulant', 'kurt', 'oscillation_map', 'powercorr', 'skew']

── 10-Node Ring Small-World ──
  ✓ metrics_10n_ring_sw_pairwise_noise.npz  keys=['DTC', 'DTC_ksg', 'K3', 'Oinfo_ksg', 'P', 'TC', 'TC_ksg', 'cumulant', 'kurt', 'osc_fraction', 'oscillation_map', 'powercorr', 'skew']
  ✓ metrics_10n_ring_sw_pairwise_diffusive_noise.npz  keys=['DTC', 'DTC_ksg', 'K3', 'Oinfo_ksg', 'P', 'TC', 'TC_ksg', 'cumulant', 'kurt', 'osc_fraction',

## 1. Metric Contour Grids (2×3)

For each information metric, a 2×3 grid compares:
- **Row 1:** 3-node models (2nd additive, 2nd diffusive, 3rd additive)
- **Row 2:** 10-node models (Ring 2nd additive, Random 2nd additive, Ring 3rd additive)

This lets us visualize how network size, topology, and coupling order affect each metric across the P × K parameter space.

In [3]:
def plot_metric_grid_2x3(metric_key, title_label):
    """Create a 2x3 Plotly contour grid for a given metric key."""

    # Row 1: 3-node models
    row1_models = [
        ('2nd-Order Additive',  data_3n),
        ('2nd-Order Diffusive', data_3n),
        ('3rd-Order Additive',  data_3n),
    ]
    # Row 2: 10-node models
    row2_models = [
        ('Ring 2nd Additive',   data_10n_ring),
        ('Random 2nd Additive', data_10n_random),
        ('Ring 3rd Additive',   data_10n_ring),
    ]

    titles = [m[0] for m in row1_models] + [m[0] for m in row2_models]
    fig = make_subplots(rows=2, cols=3, subplot_titles=titles)

    for col_idx, (model_name, model_dict) in enumerate(row1_models, 1):
        if model_name not in model_dict:
            continue
        d = model_dict[model_name]
        if metric_key not in d:
            print(f'  ⚠ {metric_key} not in {model_name}')
            continue
        fig.add_trace(go.Heatmap(
            z=d[metric_key], x=d['K3'], y=d['P'],
            colorscale='RdYlBu_r',
            showscale=(col_idx == 3),
            hovertemplate='P=%{y:.2f}<br>K=%{x:.2f}<br>' + metric_key + '=%{z:.4f}<extra></extra>'
        ), row=1, col=col_idx)

    for col_idx, (model_name, model_dict) in enumerate(row2_models, 1):
        if model_name not in model_dict:
            continue
        d = model_dict[model_name]
        if metric_key not in d:
            print(f'  ⚠ {metric_key} not in {model_name}')
            continue
        fig.add_trace(go.Heatmap(
            z=d[metric_key], x=d['K3'], y=d['P'],
            colorscale='RdYlBu_r',
            showscale=(col_idx == 3),
            hovertemplate='P=%{y:.2f}<br>K=%{x:.2f}<br>' + metric_key + '=%{z:.4f}<extra></extra>'
        ), row=2, col=col_idx)

    for c in range(1, 4):
        fig.update_xaxes(title_text='K', row=2, col=c)
    fig.update_yaxes(title_text='P', row=1, col=1)
    fig.update_yaxes(title_text='P', row=2, col=1)

    fig.update_layout(
        title_text=f'{title_label}  (3-node top  |  10-node bottom)',
        height=650, width=1100, showlegend=False
    )
    fig.show()

# ── Generate grids for all information metrics ──
metrics_to_plot = [
    ('TC',         'Total Correlation (Gaussian)'),
    ('DTC',        'Dual Total Correlation (Gaussian)'),
    ('TC_ksg',     'Total Correlation (KSG)'),
    ('DTC_ksg',    'Dual Total Correlation (KSG)'),
    ('Oinfo_ksg',  'O-information (KSG)'),
    ('cumulant',   'Cumulant'),
    ('powercorr',  'Power Correlation'),
    ('skew',       'Skewness'),
    ('kurt',       'Kurtosis'),
]

for key, label in metrics_to_plot:
    print(f'\nPlotting: {label}')
    plot_metric_grid_2x3(key, label)


Plotting: Total Correlation (Gaussian)



Plotting: Dual Total Correlation (Gaussian)



Plotting: Total Correlation (KSG)
  ⚠ TC_ksg not in 3rd-Order Additive



Plotting: Dual Total Correlation (KSG)
  ⚠ DTC_ksg not in 3rd-Order Additive



Plotting: O-information (KSG)
  ⚠ Oinfo_ksg not in 3rd-Order Additive



Plotting: Cumulant



Plotting: Power Correlation



Plotting: Skewness



Plotting: Kurtosis


## 2. P vs K Phase Diagrams — 10-Node Additive Coupling

Detailed phase diagrams for the two 10-node topologies with **2nd-order additive coupling**.

Each plot shows 6 panels: oscillation fraction, Oinfo_ksg, TC_ksg, DTC_ksg, TC (Gauss), DTC (Gauss).

In [ ]:
def plot_10n_phase_diagram(data, suptitle):
    """Plot 2x3 phase diagram for a 10-node model."""
    panels = [
        ('osc_fraction', 'Oscillation Fraction', 'YlOrRd'),
        ('Oinfo_ksg',    'O-information (KSG)',   'RdBu_r'),
        ('TC_ksg',       'TC (KSG)',              'Blues'),
        ('DTC_ksg',      'DTC (KSG)',             'Greens'),
        ('TC',           'TC (Gaussian)',          'Purples'),
        ('DTC',          'DTC (Gaussian)',         'Oranges'),
    ]

    P = data['P']
    K = data['K3']

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[p[1] for p in panels]
    )

    for idx, (key, label, cscale) in enumerate(panels):
        row = idx // 3 + 1
        col = idx % 3 + 1
        if key in data:
            fig.add_trace(go.Heatmap(
                z=data[key], x=K, y=P,
                colorscale=cscale,
                showscale=True,
                hovertemplate=f'P=%{{y:.2f}}<br>K=%{{x:.2f}}<br>{label}=%{{z:.4f}}<extra></extra>'
            ), row=row, col=col)
        else:
            print(f'  ⚠ {key} not found')

    for c in range(1, 4):
        fig.update_xaxes(title_text='K', row=2, col=c)
    fig.update_yaxes(title_text='P', row=1, col=1)
    fig.update_yaxes(title_text='P', row=2, col=1)

    fig.update_layout(
        title_text=suptitle,
        height=700, width=1100, showlegend=False
    )
    fig.show()

# ── Ring Small-World 2nd-order additive ──
if 'Ring 2nd Additive' in data_10n_ring:
    print('Ring Small-World — 2nd-Order Additive')
    plot_10n_phase_diagram(data_10n_ring['Ring 2nd Additive'],
                           '10-Node Ring Small-World: 2nd-Order Additive Coupling')

# ── Random 2nd-order additive ──
if 'Random 2nd Additive' in data_10n_random:
    print('Random — 2nd-Order Additive')
    plot_10n_phase_diagram(data_10n_random['Random 2nd Additive'],
                           '10-Node Random (ER): 2nd-Order Additive Coupling')

## 3. Combined Bifurcation Diagram — Ring vs Random

Side-by-side comparison of the oscillation map for **10-node 2nd-order additive** coupling.
Left: Ring Small-World. Right: Random (ER).
Shows how network topology reshapes the Hopf bifurcation boundary.

In [ ]:
def plot_bifurcation_comparison():
    """Side-by-side oscillation map: Ring vs Random."""

    d_ring = data_10n_ring.get('Ring 2nd Additive')
    d_rand = data_10n_random.get('Random 2nd Additive')

    if d_ring is None or d_rand is None:
        print('⚠ Cannot create comparison — data missing')
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=120)

    # Custom colormap: light blue → yellow → dark red
    from matplotlib.colors import LinearSegmentedColormap
    colors_osc = ['#e8f4f8', '#ffffcc', '#ff9900', '#cc0000', '#660000']
    cmap_osc = LinearSegmentedColormap.from_list('osc_cmap', colors_osc, N=256)

    for ax, d, title in [
        (axes[0], d_ring,  'Ring Small-World'),
        (axes[1], d_rand, 'Random (Erdős–Rényi)')
    ]:
        P, K = d['P'], d['K3']

        # Use osc_fraction if available, else oscillation_map
        osc = d.get('osc_fraction', d.get('oscillation_map'))

        im = ax.imshow(osc, origin='lower', aspect='auto',
                       extent=[K.min(), K.max(), P.min(), P.max()],
                       cmap=cmap_osc, vmin=0, vmax=1, interpolation='bilinear')

        # Draw 50% contour (bifurcation boundary)
        ax.contour(K, P, osc, levels=[0.5], colors='white', linewidths=2.5, linestyles='--')

        ax.set_xlabel('K (coupling strength)', fontsize=12)
        ax.set_ylabel('P (external drive)', fontsize=12)
        ax.set_title(f'10-Node {title}\n2nd-Order Additive', fontsize=13, fontweight='bold')

    cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.04)
    cbar.set_label('Oscillation Fraction', fontsize=11)

    plt.suptitle('Bifurcation Structure: Topology Effects on Oscillations',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../results/fig_bifurcation_ring_vs_random.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved: fig_bifurcation_ring_vs_random.png')

plot_bifurcation_comparison()

## 4. Network Topology Visualization

Aesthetic side-by-side graph of both 10-node architectures.

In [ ]:
def plot_combined_topologies():
    """Create aesthetic graph visualization of both topologies."""

    import sys
    sys.path.insert(0, '../Scripts')
    from parameters_10nodes import M_ring, M_rand, N_nodes

    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), dpi=120)

    # ── Color palette ──
    node_colors_ring  = ['#FF6B6B', '#FF8E72', '#FFA07A', '#FFB347', '#FFCC33',
                         '#87CEEB', '#6CB4EE', '#4F97D6', '#357ABD', '#2B5F8A']
    node_colors_rand  = ['#4ECDC4', '#45B7AA', '#3DA190', '#368B7A', '#2F7564',
                         '#A78BFA', '#9775E6', '#8760D2', '#774ABE', '#6735AA']

    for ax, M, title, pos_fn, node_colors, edge_style in [
        (axes[0], M_ring, 'Ring Small-World', 'circular', node_colors_ring, '-'),
        (axes[1], M_rand, 'Random (Erdős–Rényi, p=0.45)', 'spring', node_colors_rand, '-'),
    ]:
        G = nx.from_numpy_array(M)
        if pos_fn == 'circular':
            pos = nx.circular_layout(G)
        else:
            pos = nx.spring_layout(G, seed=42, k=1.5, iterations=80)

        degrees = dict(G.degree())
        node_sizes = [400 + 60 * degrees[n] for n in G.nodes()]

        # Draw edges
        nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.4, width=1.8,
                               edge_color='#888888', style=edge_style)

        # Draw nodes
        nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors[:N_nodes],
                               node_size=node_sizes, edgecolors='white', linewidths=2)

        # Draw labels
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=11, font_weight='bold',
                                font_color='white')

        avg_deg = np.mean(list(degrees.values()))
        n_edges = G.number_of_edges()
        ax.set_title(f'{title}\nNodes: {N_nodes}  |  Edges: {n_edges}  |  Avg Degree: {avg_deg:.1f}',
                     fontsize=12, fontweight='bold', pad=15)
        ax.axis('off')

    plt.suptitle('10-Node Network Architectures', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../results/fig_topologies_combined.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved: fig_topologies_combined.png')

plot_combined_topologies()

## 5. Comprehensive Statistics

Quantitative summaries for all metrics across:
- 3-node: 4 coupling architectures
- 10-node Ring & Random: 2nd and 3rd order additive

Statistics include: mean, std, min, max, oscillatory area (%), information in FP vs LC regions.

In [ ]:
def compute_model_stats(data, model_name, has_osc_fraction=False):
    """Compute statistics for a single model."""
    stats = {'model': model_name}

    # ── Oscillation ──
    osc_key = 'osc_fraction' if has_osc_fraction and 'osc_fraction' in data else 'oscillation_map'
    osc = data[osc_key]
    stats['osc_mean']  = np.nanmean(osc)
    stats['osc_std']   = np.nanstd(osc)
    stats['osc_area_%'] = 100.0 * np.sum(osc > 0.5) / osc.size

    # ── Information metrics ──
    info_keys = ['TC', 'DTC', 'TC_ksg', 'DTC_ksg', 'Oinfo_ksg', 'cumulant', 'powercorr', 'skew', 'kurt']
    for k in info_keys:
        if k in data:
            arr = data[k]
            valid = arr[~np.isnan(arr)]
            if len(valid) > 0:
                stats[f'{k}_mean'] = np.nanmean(valid)
                stats[f'{k}_std']  = np.nanstd(valid)
                stats[f'{k}_min']  = np.nanmin(valid)
                stats[f'{k}_max']  = np.nanmax(valid)

                # Stats in oscillatory vs fixed-point regions
                fp_mask = osc < 0.5
                lc_mask = osc > 0.5
                fp_vals = arr[fp_mask]
                lc_vals = arr[lc_mask]
                if len(fp_vals[~np.isnan(fp_vals)]) > 0:
                    stats[f'{k}_FP_mean'] = np.nanmean(fp_vals)
                if len(lc_vals[~np.isnan(lc_vals)]) > 0:
                    stats[f'{k}_LC_mean'] = np.nanmean(lc_vals)

    return stats

# ── 3-Node Statistics ──
print('='*70)
print('3-NODE MODEL STATISTICS')
print('='*70)

stats_3n_list = []
for name, d in data_3n.items():
    s = compute_model_stats(d, name, has_osc_fraction=False)
    stats_3n_list.append(s)
    print(f'\n{name}:')
    print(f'  Oscillatory area: {s["osc_area_%"]:.1f}%')
    for k in ['TC_ksg', 'DTC_ksg', 'Oinfo_ksg']:
        if f'{k}_mean' in s:
            print(f'  {k}: {s[f"{k}_mean"]:.6f} ± {s[f"{k}_std"]:.6f}  '
                  f'[{s[f"{k}_min"]:.6f}, {s[f"{k}_max"]:.6f}]')
            if f'{k}_FP_mean' in s and f'{k}_LC_mean' in s:
                print(f'    FP region: {s[f"{k}_FP_mean"]:.6f}  |  LC region: {s[f"{k}_LC_mean"]:.6f}')

df_3n = pd.DataFrame(stats_3n_list).set_index('model')

# ── 10-Node Statistics ──
print('\n' + '='*70)
print('10-NODE MODEL STATISTICS')
print('='*70)

stats_10n_list = []
all_10n = {}
all_10n.update(data_10n_ring)
all_10n.update(data_10n_random)

# Focus on additive models
models_10n_focus = [
    'Ring 2nd Additive', 'Ring 3rd Additive',
    'Random 2nd Additive', 'Random 3rd Additive',
]

for name in models_10n_focus:
    if name not in all_10n:
        continue
    d = all_10n[name]
    s = compute_model_stats(d, name, has_osc_fraction=True)
    stats_10n_list.append(s)
    print(f'\n{name}:')
    print(f'  Oscillatory area: {s["osc_area_%"]:.1f}%')
    for k in ['TC_ksg', 'DTC_ksg', 'Oinfo_ksg']:
        if f'{k}_mean' in s:
            print(f'  {k}: {s[f"{k}_mean"]:.6f} ± {s[f"{k}_std"]:.6f}  '
                  f'[{s[f"{k}_min"]:.6f}, {s[f"{k}_max"]:.6f}]')
            if f'{k}_FP_mean' in s and f'{k}_LC_mean' in s:
                print(f'    FP region: {s[f"{k}_FP_mean"]:.6f}  |  LC region: {s[f"{k}_LC_mean"]:.6f}')

df_10n = pd.DataFrame(stats_10n_list).set_index('model')

print('\n✓ Statistics computed')

In [ ]:
# ── Display summary tables ──
print('\n' + '='*70)
print('3-NODE SUMMARY TABLE')
print('='*70)
cols_display = [c for c in df_3n.columns if 'mean' in c or 'area' in c]
print(df_3n[cols_display].round(6).to_string())

print('\n' + '='*70)
print('10-NODE SUMMARY TABLE')
print('='*70)
cols_display_10 = [c for c in df_10n.columns if 'mean' in c or 'area' in c]
print(df_10n[cols_display_10].round(6).to_string())

# ── Export to CSV ──
df_3n.to_csv('../results/statistics_3node_summary.csv')
df_10n.to_csv('../results/statistics_10node_summary.csv')

print('\n✓ Exported: statistics_3node_summary.csv')
print('✓ Exported: statistics_10node_summary.csv')

## 6. Comparative Analysis

### Topology Effects: Ring vs Random
### Coupling Order Effects: 2nd vs 3rd Order
### KSG vs Gaussian Estimator Comparison

In [ ]:
def compare_two_models(d1, name1, d2, name2, title):
    """Print comparison between two models."""
    print(f'\n{"="*70}')
    print(f'{title}')
    print(f'{"="*70}')
    print(f'  {"Metric":<15} {"":>12} {"":>12} {"Diff":>10} {"% Change":>10}')
    print(f'  {"":15} {name1:>12} {name2:>12}')
    print(f'  {"-"*60}')

    for k in ['TC_ksg', 'DTC_ksg', 'Oinfo_ksg', 'TC', 'DTC', 'cumulant', 'powercorr']:
        if k in d1 and k in d2:
            m1 = np.nanmean(d1[k])
            m2 = np.nanmean(d2[k])
            diff = m2 - m1
            pct = 100 * diff / (abs(m1) + 1e-10)
            print(f'  {k:<15} {m1:>12.6f} {m2:>12.6f} {diff:>+10.6f} {pct:>+9.1f}%')

    # Oscillation area comparison
    osc1 = d1.get('osc_fraction', d1.get('oscillation_map'))
    osc2 = d2.get('osc_fraction', d2.get('oscillation_map'))
    area1 = 100 * np.sum(osc1 > 0.5) / osc1.size
    area2 = 100 * np.sum(osc2 > 0.5) / osc2.size
    print(f'  {"osc_area %":<15} {area1:>12.1f} {area2:>12.1f} {area2-area1:>+10.1f} {"":>10}')

# ── Topology comparison: Ring vs Random (2nd additive) ──
d_ring_2 = data_10n_ring.get('Ring 2nd Additive')
d_rand_2 = data_10n_random.get('Random 2nd Additive')
if d_ring_2 and d_rand_2:
    compare_two_models(d_ring_2, 'Ring', d_rand_2, 'Random',
                       'TOPOLOGY EFFECTS: Ring vs Random (2nd-Order Additive)')

# ── Order comparison: 2nd vs 3rd (Ring additive) ──
d_ring_3 = data_10n_ring.get('Ring 3rd Additive')
if d_ring_2 and d_ring_3:
    compare_two_models(d_ring_2, '2nd-Order', d_ring_3, '3rd-Order',
                       'COUPLING ORDER EFFECTS: 2nd vs 3rd (Ring Additive)')

# ── KSG vs Gaussian ──
print(f'\n{"="*70}')
print('KSG vs GAUSSIAN ESTIMATOR COMPARISON')
print(f'{"="*70}')

for name, d in list(data_3n.items()) + list(data_10n_ring.items()) + list(data_10n_random.items()):
    if 'TC_ksg' in d and 'TC' in d:
        tc_corr = np.corrcoef(d['TC'].flatten(), d['TC_ksg'].flatten())[0,1]
        dtc_corr = np.corrcoef(d['DTC'].flatten(), d['DTC_ksg'].flatten())[0,1] if 'DTC_ksg' in d else float('nan')
        print(f'  {name:<25}  TC corr={tc_corr:.4f}  DTC corr={dtc_corr:.4f}')

print('\n✓ Comparative analysis complete')

## 7. Bifurcation Boundary Extraction

Quantitative extraction of Hopf bifurcation parameters for each model.

In [ ]:
def extract_bifurcation_stats(data, model_name, use_osc_frac=False):
    """Extract bifurcation boundary statistics."""
    osc = data.get('osc_fraction' if use_osc_frac else 'oscillation_map',
                   data.get('oscillation_map'))
    P = data['P']
    K = data['K3']

    # Find boundary: for each P, find minimum K where osc > 0.5
    boundary_K = []
    boundary_P = []

    for i in range(len(P)):
        row = osc[i, :]
        osc_indices = np.where(row > 0.5)[0]
        if len(osc_indices) > 0:
            j_onset = osc_indices[0]
            # Interpolate between j_onset-1 and j_onset
            if j_onset > 0:
                k_low = K[j_onset - 1]
                k_high = K[j_onset]
                v_low = row[j_onset - 1]
                v_high = row[j_onset]
                if v_high != v_low:
                    k_interp = k_low + (0.5 - v_low) / (v_high - v_low) * (k_high - k_low)
                else:
                    k_interp = k_high
                boundary_K.append(k_interp)
            else:
                boundary_K.append(K[0])
            boundary_P.append(P[i])

    if not boundary_K:
        print(f'  {model_name}: No bifurcation detected')
        return None

    boundary_K = np.array(boundary_K)
    boundary_P = np.array(boundary_P)

    stats = {
        'model': model_name,
        'K_onset_min': np.min(boundary_K),
        'K_onset_max': np.max(boundary_K),
        'K_onset_mean': np.mean(boundary_K),
        'P_range': f'[{boundary_P.min():.2f}, {boundary_P.max():.2f}]',
        'n_boundary_points': len(boundary_K),
        'osc_area_%': 100 * np.sum(osc > 0.5) / osc.size,
    }

    print(f'\n  {model_name}:')
    print(f'    K onset: {stats["K_onset_min"]:.4f} – {stats["K_onset_max"]:.4f} (mean {stats["K_onset_mean"]:.4f})')
    print(f'    P range with oscillation: {stats["P_range"]}')
    print(f'    Oscillatory area: {stats["osc_area_%"]:.1f}%')
    print(f'    Boundary points: {stats["n_boundary_points"]}')

    return stats

print('='*70)
print('BIFURCATION BOUNDARY EXTRACTION')
print('='*70)

bifurc_stats = []

# 3-node models
print('\n── 3-Node Models ──')
for name, d in data_3n.items():
    s = extract_bifurcation_stats(d, name, use_osc_frac=False)
    if s:
        bifurc_stats.append(s)

# 10-node models
print('\n── 10-Node Ring Small-World ──')
for name in ['Ring 2nd Additive', 'Ring 3rd Additive']:
    if name in data_10n_ring:
        s = extract_bifurcation_stats(data_10n_ring[name], name, use_osc_frac=True)
        if s:
            bifurc_stats.append(s)

print('\n── 10-Node Random ──')
for name in ['Random 2nd Additive', 'Random 3rd Additive']:
    if name in data_10n_random:
        s = extract_bifurcation_stats(data_10n_random[name], name, use_osc_frac=True)
        if s:
            bifurc_stats.append(s)

# Summary table
df_bifurc = pd.DataFrame(bifurc_stats).set_index('model')
print('\n' + '='*70)
print('BIFURCATION SUMMARY TABLE')
print('='*70)
print(df_bifurc.to_string())

df_bifurc.to_csv('../results/bifurcation_summary.csv')
print('\n✓ Exported: bifurcation_summary.csv')